<a href="https://colab.research.google.com/github/Numanur/heart-failure-monitoring-llm-rag/blob/main/Thesis_2_state_to_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pandas as pd
import numpy as np
from datetime import datetime

PROJECT_DIR = "/content/drive/MyDrive/llm"

STATE_DIR = f"{PROJECT_DIR}/patient_states"
PROCESSED_DIR = f"{PROJECT_DIR}/data_processed"
OUTPUT_DIR = f"{PROJECT_DIR}/llm_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

FULL_STATE_PATH = f"{STATE_DIR}/patient_state_text.csv"
COMPACT_STATE_PATH = f"{STATE_DIR}/patient_state_compact.csv"
PATIENT_STATES_JSON_PATH = f"{STATE_DIR}/patient_states.json"
PATIENT_LABELS_PATH = f"{PROCESSED_DIR}/patient_labels.csv"

print("Project directory:", PROJECT_DIR)
print("Patient-state text path:", FULL_STATE_PATH)
print("Compact patient-state path:", COMPACT_STATE_PATH)
print("Output directory:", OUTPUT_DIR)

Mounted at /content/drive
Project directory: /content/drive/MyDrive/llm
Patient-state text path: /content/drive/MyDrive/llm/patient_states/patient_state_text.csv
Compact patient-state path: /content/drive/MyDrive/llm/patient_states/patient_state_compact.csv
Output directory: /content/drive/MyDrive/llm/llm_outputs


In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio transformers accelerate pandas

In [ ]:
patient_state_text_df = pd.read_csv(FULL_STATE_PATH)
patient_state_compact_df = pd.read_csv(COMPACT_STATE_PATH)

print("Full patient-state file shape:", patient_state_text_df.shape)
print("Compact patient-state file shape:", patient_state_compact_df.shape)

display(patient_state_text_df.head())
display(patient_state_compact_df.head())

# Labels are loaded only for later offline evaluation.
# Do NOT put labels into the LLM prompt.
if os.path.exists(PATIENT_LABELS_PATH):
    patient_labels_df = pd.read_csv(PATIENT_LABELS_PATH)
    print("Patient labels file shape:", patient_labels_df.shape)
    display(patient_labels_df.head())
else:
    patient_labels_df = None
    print("patient_labels.csv not found. This is okay for generation, but needed for later evaluation.")

In [ ]:
def normalize_patient_id(x):
    """
    Converts patient IDs into stable string format.
    This avoids problems like 123 vs 123.0.
    """
    try:
        return str(int(float(x)))
    except Exception:
        return str(x).strip()


patient_state_text_df["patient_id_norm"] = patient_state_text_df["inpatient.number"].apply(normalize_patient_id)
patient_state_compact_df["patient_id_norm"] = patient_state_compact_df["inpatient.number"].apply(normalize_patient_id)

FULL_PATIENT_CONTEXT = dict(
    zip(
        patient_state_text_df["patient_id_norm"],
        patient_state_text_df["patient_state_text"]
    )
)

COMPACT_PATIENT_CONTEXT = dict(
    zip(
        patient_state_compact_df["patient_id_norm"],
        patient_state_compact_df["patient_state_compact"]
    )
)

ALL_PATIENT_IDS = sorted(
    FULL_PATIENT_CONTEXT.keys(),
    key=lambda x: int(x) if str(x).isdigit() else str(x)
)

print("Number of full patient contexts:", len(FULL_PATIENT_CONTEXT))
print("Number of compact patient contexts:", len(COMPACT_PATIENT_CONTEXT))
print("First 10 patient IDs:", ALL_PATIENT_IDS[:10])

sample_patient_id = ALL_PATIENT_IDS[0]

print("\nSample patient ID:", sample_patient_id)
print("\nSample full patient context:\n")
print(FULL_PATIENT_CONTEXT[sample_patient_id][:2000])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. In Colab, go to Runtime > Change runtime type > GPU.")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

DEVICE = next(model.parameters()).device

print("Model loaded.")
print("Model device:", DEVICE)
print("Tokenizer pad token:", tokenizer.pad_token)

In [ ]:
def generate_stream_chunks(user_prompt, max_new_tokens=500):
    """
    Streams generated text from the model.

    Input:
    - user_prompt: final prompt string containing instruction, patient context, and question.
    - max_new_tokens: maximum new tokens to generate.

    Output:
    - yields text chunks.
    """

    messages = [
        {"role": "user", "content": user_prompt}
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    model_inputs = {k: v.to(DEVICE) for k, v in model_inputs.items()}

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generation_kwargs = dict(
        **model_inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    for new_text in streamer:
        yield new_text

In [ ]:
test_prompt = "Explain retrieval augmented generation in simple words."

for chunk in generate_stream_chunks(test_prompt, max_new_tokens=80):
    print(chunk, end="", flush=True)

In [ ]:
def build_llm_only_prompt(patient_context, user_question):
    prompt = f"""
You are a cautious medical AI research assistant for an offline retrospective heart-failure study.

Important setting:
- This is not a real patient consultation.
- This is not a live clinical deployment.
- You are analyzing a retrospective post-discharge heart-failure patient case.
- No external guideline evidence has been retrieved yet.
- Therefore, do not claim that your answer is guideline-grounded.
- Use only the patient context provided below and general clinical reasoning.
- Do not invent missing patient facts.
- If important information is missing, clearly say that it is missing.
- Do not provide a diagnosis.
- Do not provide emergency instructions as if you are replacing a clinician.
- Give cautious monitoring-oriented suggestions.
- Avoid overconfident statements.
- Do not mention the hidden outcome label or make claims about known readmission/death status.

Patient context:
{patient_context}

User question:
{user_question}

Required response format:
1. Patient-specific summary:
2. Relevant risk or deterioration cues:
3. Cautious monitoring suggestion:
4. Missing information / uncertainty:
5. Safety note:

Answer:
""".strip()

    return prompt

In [ ]:
def get_patient_context(patient_id, context_mode="full"):
    """
    Returns patient context text for a selected patient.

    context_mode:
    - 'full' uses patient_state_text.csv
    - 'compact' uses patient_state_compact.csv
    """

    patient_id_norm = normalize_patient_id(patient_id)

    if context_mode == "compact":
        context_dict = COMPACT_PATIENT_CONTEXT
    else:
        context_dict = FULL_PATIENT_CONTEXT

    if patient_id_norm not in context_dict:
        raise ValueError(f"Patient ID not found: {patient_id}")

    return context_dict[patient_id_norm]

In [ ]:
sample_patient_id = ALL_PATIENT_IDS[0]

sample_question = "What post-discharge monitoring issues are relevant for this heart-failure patient?"

patient_context = get_patient_context(sample_patient_id, context_mode="full")

prompt = build_llm_only_prompt(
    patient_context=patient_context,
    user_question=sample_question
)

print(prompt[:4000])

In [ ]:
for chunk in generate_stream_chunks(prompt, max_new_tokens=500):
    print(chunk, end="", flush=True)

In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="HF Patient-Context LLM Baseline API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
def home():
    return {
        "status": "ok",
        "message": "HF patient-context LLM baseline API is running",
        "rag_used": False,
        "available_endpoints": ["/patients", "/generate", "/generate_patient"]
    }


@app.get("/patients")
def list_patients(limit: int = 20):
    limit = max(1, min(int(limit), 100))

    return {
        "total_patients": len(ALL_PATIENT_IDS),
        "returned": limit,
        "patient_ids": ALL_PATIENT_IDS[:limit]
    }


@app.post("/generate")
async def generate(request: Request):
    """
    Raw prompt endpoint.
    Use this for simple testing.
    """

    body = await request.json()
    prompt = body.get("prompt", "").strip()
    max_new_tokens = int(body.get("max_new_tokens", 300))

    if not prompt:
        return JSONResponse(
            status_code=400,
            content={"error": "Prompt is empty"}
        )

    def stream_response():
        try:
            for chunk in generate_stream_chunks(prompt, max_new_tokens=max_new_tokens):
                yield chunk
        except Exception as e:
            print("Generation error:", str(e))
            yield f"\n[ERROR] {str(e)}"

    return StreamingResponse(
        stream_response(),
        media_type="text/plain",
        headers={"Cache-Control": "no-cache"},
    )


@app.post("/generate_patient")
async def generate_patient(request: Request):
    """
    Patient-specific endpoint.

    Input JSON:
    {
        "patient_id": "...",
        "question": "...",
        "context_mode": "full" or "compact",
        "max_new_tokens": 500,
        "save_output": true
    }
    """

    body = await request.json()

    patient_id = body.get("patient_id", None)
    question = body.get("question", "").strip()
    context_mode = body.get("context_mode", "full")
    max_new_tokens = int(body.get("max_new_tokens", 500))
    save_output = bool(body.get("save_output", False))

    if patient_id is None:
        return JSONResponse(
            status_code=400,
            content={"error": "patient_id is required"}
        )

    if not question:
        return JSONResponse(
            status_code=400,
            content={"error": "question is required"}
        )

    if context_mode not in ["full", "compact"]:
        return JSONResponse(
            status_code=400,
            content={"error": "context_mode must be either 'full' or 'compact'"}
        )

    patient_id_norm = normalize_patient_id(patient_id)

    try:
        patient_context = get_patient_context(patient_id_norm, context_mode=context_mode)
    except ValueError:
        return JSONResponse(
            status_code=404,
            content={
                "error": "patient_id not found",
                "patient_id": patient_id
            }
        )

    prompt = build_llm_only_prompt(
        patient_context=patient_context,
        user_question=question
    )

    def stream_response():
        full_output = ""

        try:
            for chunk in generate_stream_chunks(prompt, max_new_tokens=max_new_tokens):
                full_output += chunk
                yield chunk

            if save_output:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                output_path = f"{OUTPUT_DIR}/api_llm_only_output_{patient_id_norm}_{timestamp}.json"

                result = {
                    "patient_id": patient_id_norm,
                    "question": question,
                    "context_mode": context_mode,
                    "prompt_type": "llm_only_patient_context_baseline",
                    "rag_used": False,
                    "prompt": prompt,
                    "model_output": full_output,
                    "created_at": datetime.now().isoformat()
                }

                with open(output_path, "w") as f:
                    json.dump(result, f, indent=2)

                print("Saved API output:", output_path)

        except Exception as e:
            print("Generation error:", str(e))
            yield f"\n[ERROR] {str(e)}"

    return StreamingResponse(
        stream_response(),
        media_type="text/plain",
        headers={"Cache-Control": "no-cache"},
    )

In [ ]:
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

print("Server started on port 8000")

Server started on port 8000


In [ ]:
from pyngrok import ngrok

# ngrok.set_auth_token("3CenhdMC9xr7r8lyEVTrEJ9wwZ5_6CE9ZR6vzT8ZSxsaV6bVd")

ngrok.kill()

tunnel = ngrok.connect(8000, bind_tls=True)
public_url = tunnel.public_url

print("Public URL:", public_url)
print("Raw prompt endpoint:", public_url + "/generate")
print("Patient endpoint:", public_url + "/generate_patient")
print("Patient list endpoint:", public_url + "/patients")

INFO:     Started server process [2716]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: https://worst-cogwheel-recolor.ngrok-free.dev
Raw prompt endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/generate
Patient endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/generate_patient
Patient list endpoint: https://worst-cogwheel-recolor.ngrok-free.dev/patients


In [ ]:
import requests

patients_response = requests.get(
    public_url + "/patients",
    params={"limit": 10}
)

print("Status code:", patients_response.status_code)
print(patients_response.json())

In [ ]:
url = f"{public_url}/generate"

response = requests.post(
    url,
    json={
        "prompt": "Tell me in simple words what an embedding is.",
        "max_new_tokens": 120
    },
    stream=True
)

print("Status code:", response.status_code)
print("Streaming output:\n")

for chunk in response.iter_content(chunk_size=None, decode_unicode=True):
    if chunk:
        print(chunk, end="", flush=True)

In [ ]:
sample_patient_id = ALL_PATIENT_IDS[0]

url = f"{public_url}/generate_patient"

response = requests.post(
    url,
    json={
        "patient_id": sample_patient_id,
        "question": "What post-discharge monitoring issues are relevant for this heart-failure patient?",
        "context_mode": "full",
        "max_new_tokens": 500,
        "save_output": True
    },
    stream=True
)

print("Sample patient ID:", sample_patient_id)
print("Status code:", response.status_code)
print("Streaming output:\n")

for chunk in response.iter_content(chunk_size=None, decode_unicode=True):
    if chunk:
        print(chunk, end="", flush=True)

In [ ]:
sample_patient_id = ALL_PATIENT_IDS[0]

response = requests.post(
    f"{public_url}/generate_patient",
    json={
        "patient_id": sample_patient_id,
        "question": "Summarize this patient's main monitoring priorities after discharge.",
        "context_mode": "compact",
        "max_new_tokens": 400,
        "save_output": True
    },
    stream=True
)

print("Sample patient ID:", sample_patient_id)
print("Status code:", response.status_code)
print("Streaming output:\n")

for chunk in response.iter_content(chunk_size=None, decode_unicode=True):
    if chunk:
        print(chunk, end="", flush=True)

In [ ]:
r = requests.options(
    public_url + "/generate_patient",
    headers={
        "Origin": "http://localhost:5173",
        "Access-Control-Request-Method": "POST",
        "Access-Control-Request-Headers": "content-type",
    },
)

print("status:", r.status_code)
print("allow-origin:", r.headers.get("access-control-allow-origin"))
print("allow-methods:", r.headers.get("access-control-allow-methods"))
print("allow-headers:", r.headers.get("access-control-allow-headers"))
print("all headers:", dict(r.headers))

In [ ]:
example_frontend_payload = {
    "patient_id": sample_patient_id,
    "question": "What should be monitored after discharge?",
    "context_mode": "full",
    "max_new_tokens": 500,
    "save_output": True
}

print("Frontend should POST this JSON to:")
print(public_url + "/generate_patient")

print("\nExample payload:")
print(json.dumps(example_frontend_payload, indent=2))